In [1]:
import pandas as pd
import anthropic
import json
import time
import os
from dotenv import load_dotenv
import time
from IPython.display import HTML

# Load sample csv

In [2]:
df_sample = pd.read_csv("gold_standard_sample.csv.gz", compression="gzip")
print(f"Loaded: {len(df_sample)} judgments")

Loaded: 1000 judgments


# Annotatie Anthropic Haiku


In [3]:
# ── Prompts ───────────────────────────────────────────────────────────────────

USER_PROMPT_TEMPLATE = "Annotate all legal references in the following Dutch court judgment:\n\n{text}"

SYSTEM_PROMPT = """You are an expert annotator of Dutch legal texts. Your task is to identify all legal references in a court judgment, following Dutch legal citation standards.

You must identify references of exactly four types:

BWB — Dutch legislation
- Full law names: "Wetboek van Strafrecht", "Algemene wet bestuursrecht"
- Abbreviations: Awb, BW, Sr, Sv, Rv, Gw, Wro, Mw, EVRM etc.
- Article references: "art. 6:162 BW", "artikel 8 lid 1 onderdeel b Awb"
- Range references: "art. 17-23 Mw" (ONE span)
- Book/chapter/section references: "hoofdstuk 3 Wro", "titel 1 afd. 2 Boek 3 BW"
- Multiple laws in one reference: "art. 2.14a lid 3 Wet IB 2001 en art. 16 lid 3 SW 1956" (TWO spans)

ECLI — References to court decisions
- ECLI numbers: "ECLI:NL:HR:2023:847"
- Court + date: "HR 12 maart 2020", "ABRvS 16 maart 2016"
- Court + date + ECLI: "HR 29 januari 2016, ECLI:NL:HR:2016:162"
- Case names (roepnamen): "het arrest Kühne Heitz"
- Journal citations: "NJ 2020/123", "AB 2019/45"
- Case numbers + court name: "RvBAz 2014/71419" (even without ECLI number)
- With paragraph reference: "HR 29 januari 2016, ECLI:NL:HR:2016:162, r.o. 3.8" (ONE span)

CELEX — European legislation
- Full names: "Richtlijn 2011/95/EU", "Verordening (EU) nr. 1215/2012"
- CELEX numbers: "CELEX:32012R1215"
- Article references: "art. 7 Verordening (EU) nr. 1215/2012" (ONE span)
- Informal names: "de Brussel I bis-verordening"

OEP — Official publications
- Staatsblad: "Stb. 2023, 506", "Staatsblad 2023, 506"
- Staatscourant: "Stcrt. 2020, 12345"
- Kamerstukken: "Kamerstukken II 1997/98, 25 900, nr. 6"
- Handelingen: "Handelingen II 2020/21, nr. 45, item 3"

ANNOTATION RULES:
- Be recall-oriented: when in doubt, include the span. Missing a reference is worse than a false positive.
- Copy spans EXACTLY as they appear in the text — character for character.
- Span boundaries: start at the first meaningful word, end at the last. Do NOT include leading articles or prepositions (e.g. "het", "de", "van").
- Article reference + law name = ONE span: "art. 6:162 BW" is one BWB span.
- "art. 6 en 7 EVRM" = TWO spans: "art. 6 EVRM" and "art. 7 EVRM".
- Local aliases: if the text defines a shorthand (e.g. "Algemene wet bestuursrecht (hierna: de Awb)"), annotate all later occurrences of that alias with the correct label.
- Abbreviations alone (e.g. "Awb", "BW", "Sr") are valid BWB spans even without an article number.

DO NOT annotate:
- Party names, lawyer names, judge names
- Internal references: "zie overweging 3.2", "zoals hierboven overwogen"
- General terms without specific reference: "de wet", "het verdrag", "de richtlijn"
- Dates and case numbers that are NOT part of a legal reference
- Document titles of the judgment itself

Return ONLY a JSON array. Each item has exactly two fields:
- "span": the exact text copied from the judgment
- "label": one of BWB, ECLI, CELEX, OEP

Example output:
[
  {"span": "artikel 7:671b lid 1, onderdeel a, van het Burgerlijk Wetboek", "label": "BWB"},
  {"span": "ECLI:NL:HR:2023:847", "label": "ECLI"},
  {"span": "Richtlijn 2011/95/EU", "label": "CELEX"},
  {"span": "Stb. 2023, 506", "label": "OEP"},
  {"span": "RvBAz 2014/71419", "label": "ECLI"}
]

If no legal references are found, return an empty array: []"""

In [4]:
load_dotenv()

# Haal de API key op
claude_key = os.getenv("claude_key")

os.environ["ANTHROPIC_API_KEY"] = claude_key
client = anthropic.Anthropic()

# ── Hulpfuncties ──────────────────────────────────────────────────────────────

def zoek_offsets(tekst, annotations):
    """Zoekt offsets van spans in de tekst."""
    resultaat = []
    zoekpositie = 0

    for ann in annotations:
        if not isinstance(ann, dict) or "span" not in ann or "label" not in ann:
            continue

        span = ann["span"]

        pos = tekst.find(span, zoekpositie)
        if pos != -1:
            resultaat.append({"span": span, "label": ann["label"], "start": pos, "end": pos + len(span)})
            zoekpositie = pos + 1
            continue

        # Fallback zonder zoekpositie restrictie
        pos = tekst.find(span)
        if pos != -1:
            resultaat.append({"span": span, "label": ann["label"], "start": pos, "end": pos + len(span)})
            continue

        print(f"  ⚠ Span niet gevonden: '{span}'")

    return resultaat


def chunk_tekst(tekst, max_chars=24000):
    """Splits tekst in stukken op alinea-grenzen."""
    if len(tekst) <= max_chars:
        return [tekst]

    paragraphs = tekst.split("\n\n")
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) > max_chars:
            if current:
                chunks.append(current.strip())
            current = para
        else:
            current += "\n\n" + para
    if current:
        chunks.append(current.strip())
    return chunks


def annotate_judgment(tekst):
    """Annoteert een uitspraak via Claude Haiku."""
    chunks = chunk_tekst(tekst)
    all_annotations = []
    offset = 0

    for chunk in chunks:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(text=chunk)
            }]
        )

        raw = response.content[0].text.strip()

        # Verwijder markdown als het model die toch toevoegt
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]

        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                annotations = parsed
            elif isinstance(parsed, dict):
                annotations = next((v for v in parsed.values() if isinstance(v, list)), [])
            else:
                annotations = []

            annotations_met_offsets = zoek_offsets(chunk, annotations)
            for ann in annotations_met_offsets:
                ann["start"] += offset
                ann["end"] += offset
            all_annotations.extend(annotations_met_offsets)

        except json.JSONDecodeError as e:
            print(f"  Parse error op offset {offset}: {e}")

        offset += len(chunk) + 2
        time.sleep(0.3)

    return all_annotations


# ── Hoofdloop ─────────────────────────────────────────────────────────────────

OUTPUT_FILE = "gold_standard_annotations.json"

# Laad al geannoteerde ECLIs zodat je kunt hervatten na onderbreking
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        results = json.load(f)
    al_gedaan = {r["ecli"] for r in results}
    print(f"Hervat: {len(al_gedaan)} uitspraken al gedaan")
else:
    results = []
    al_gedaan = set()

failed = []

for i, row in df_sample.head(20).iterrows():
    if row["ecli"] in al_gedaan:
        continue  # Sla al geannoteerde uitspraken over

    try:
        annotations = annotate_judgment(row["text"])
        results.append({
            "ecli": row["ecli"],
            "annotations": annotations,
            "n_annotations": len(annotations)
        })
        al_gedaan.add(row["ecli"])

        if len(al_gedaan) % 10 == 0:
            print(f"{len(al_gedaan)}/{len(df_sample)} — laatste: {len(annotations)} annotaties")

        # Sla tussentijds op na elke 10 uitspraken
        if len(al_gedaan) % 10 == 0:
            with open(OUTPUT_FILE, "w") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

        time.sleep(0.5)

    except Exception as e:
        print(f"Fout bij {row['ecli']}: {e}")
        failed.append(row["ecli"])

# Sla definitief op
with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nKlaar. {len(results)} uitspraken geannoteerd, {len(failed)} mislukt.")
if failed:
    print(f"Mislukt: {failed}")

Hervat: 20 uitspraken al gedaan

Klaar. 20 uitspraken geannoteerd, 0 mislukt.


# Annotaties checken

In [57]:
def toon_annotaties(ecli, results, df_sample):
    result = next(r for r in results if r["ecli"] == ecli)
    tekst = df_sample[df_sample["ecli"] == ecli]["text"].iloc[0]
    
    kleuren = {
        "BWB":   "#fff388f4",
        "ECLI":  "#90caf9",
        "CELEX": "#a5d6a7",
        "OEP":   "#ffcc80"
    }
    
    # Iets donkerdere tint voor duplicaten
    kleuren_duplicate = {
        "BWB":   "#e1ce04",
        "ECLI":  "#4a90d9",
        "CELEX": "#66bb6a",
        "OEP":   "#fd9226"
    }
    
    anns = sorted(result["annotations"], key=lambda x: (x["start"], -x["end"]))
    
    html = "<style>mark { padding: 2px; border-radius: 3px; }</style>"
    html += "<p style='font-family: Arial; line-height: 1.8;'>"
    
    positie = 0
    for ann in anns:
        if ann["start"] < positie:
            continue
        html += tekst[positie:ann["start"]]
        
        is_dup = ann.get("is_duplicate", False)
        kleur = (kleuren_duplicate if is_dup else kleuren).get(ann["label"], "#e0e0e0")
        tekst_kleur = "white" if is_dup else "black"
        label_suffix = " ★" if is_dup else ""
        
        html += (f'<mark style="background:{kleur}; color:{tekst_kleur}" '
                 f'title="{ann["label"]}{label_suffix}">'
                 f'{tekst[ann["start"]:ann["end"]]}</mark>')
        positie = ann["end"]
    
    html += tekst[positie:] + "</p>"
    
    # Legenda met beide varianten
    legenda_items = []
    for label, kleur in kleuren.items():
        legenda_items.append(f'<span style="background:{kleur}; color:black; padding:3px 6px; border-radius:3px">{label}</span>')
    for label, kleur in kleuren_duplicate.items():
        legenda_items.append(f'<span style="background:{kleur}; color:white; padding:3px 6px; border-radius:3px">{label} ★</span>')
    legenda = " ".join(legenda_items)
    
    return HTML(f"<b>{ecli}</b> — {result['n_annotations']} annotaties<br>{legenda}<br><br>{html}")

In [58]:
with open("gold_standard_annotations_recovered.json") as f:
    results = json.load(f)

eclis = [r["ecli"] for r in results]
print(f"Totaal: {len(eclis)} uitspraken")
# print(eclis)

Totaal: 1000 uitspraken


In [50]:
for ecli in eclis[:1]:
    display(toon_annotaties(ecli, results, df_sample))

## Duplicates toevoegen

In [ ]:
def verwijder_overlappen(annotations):
    """Bewaart bij overlappende spans altijd de langste."""
    if not annotations:
        return []
    
    # Sorteer op startpositie, langste span eerst bij gelijke start
    gesorteerd = sorted(annotations, key=lambda x: (x["start"], -(x["end"])))
    
    resultaat = []
    laatste_end = -1
    
    for ann in gesorteerd:
        if ann["start"] >= laatste_end:
            # Geen overlap — voeg toe
            resultaat.append(ann)
            laatste_end = ann["end"]
        else:
            # Overlap — vervang vorige als deze langer is
            if ann["end"] > laatste_end:
                resultaat[-1] = ann
                laatste_end = ann["end"]
    
    return resultaat

def voeg_duplicaten_toe(tekst, annotations):
    # Bewaar originele posities
    originele_starts = {a["start"] for a in annotations}
    
    lokale_gazetteer = {}
    for ann in annotations:
        if ann["span"] not in lokale_gazetteer:
            lokale_gazetteer[ann["span"]] = ann["label"]
    
    alle_annotations = []
    for span, label in lokale_gazetteer.items():
        pos = 0
        while True:
            pos = tekst.find(span, pos)
            if pos == -1:
                break
            alle_annotations.append({
                "span": span,
                "label": label,
                "start": pos,
                "end": pos + len(span),
                "is_duplicate": pos not in originele_starts  # Nieuw veld
            })
            pos += 1
    
    alle_annotations.sort(key=lambda x: x["start"])
    return verwijder_overlappen(alle_annotations)

In [60]:
for result in results:
    ecli = result["ecli"]
    tekst = df_sample[df_sample["ecli"] == ecli]["text"].iloc[0]
    
    uitgebreid = voeg_duplicaten_toe(tekst, result["annotations"])
    result["annotations"] = uitgebreid
    result["n_annotations"] = len(uitgebreid)

print(f"Klaar. Totaal annotaties: {sum(r['n_annotations'] for r in results)}")

# Sla op
with open("gold_standard_annotations_duplicate_ext.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

Klaar. Totaal annotaties: 7155


In [55]:
totaal_spans = 0
totaal_extra = 0
uitspraken_met_extra = 0
gemiste_spans = Counter()

def is_gedekt(pos, span, annotations):
    """Checkt of een positie al gedekt wordt door een bestaande annotatie."""
    eind = pos + len(span)
    for ann in annotations:
        if ann["start"] <= pos and ann["end"] >= eind:
            return True
    return False

for result in results:
    ecli = result["ecli"]
    tekst = df_sample[df_sample["ecli"] == ecli]["text"].iloc[0]
    
    lokale_gazetteer = {}
    for ann in result["annotations"]:
        if ann["span"] not in lokale_gazetteer:
            lokale_gazetteer[ann["span"]] = ann["label"]
    
    extra_deze_uitspraak = 0
    
    for span, label in lokale_gazetteer.items():
        alle_voorkomens = []
        pos = 0
        while True:
            pos = tekst.find(span, pos)
            if pos == -1:
                break
            alle_voorkomens.append(pos)
            totaal_spans += 1
            pos += 1
        
        niet_gedekt = [p for p in alle_voorkomens 
                       if not is_gedekt(p, span, result["annotations"])]
        
        extra_deze_uitspraak += len(niet_gedekt)
        for p in niet_gedekt:
            gemiste_spans[span] += 1
    
    if extra_deze_uitspraak > 0:
        uitspraken_met_extra += 1
        totaal_extra += extra_deze_uitspraak

print(f"Totaal voorkomens van geannoteerde spans: {totaal_spans}")
print(f"Niet geannoteerde duplicaten:             {totaal_extra}")
print(f"Uitspraken met minstens één duplicaat:    {uitspraken_met_extra}/{len(results)}")
print(f"Gemiddeld extra per uitspraak:            {totaal_extra/len(results):.1f}")

print("\nTop 20 meest gemiste spans:")
for span, count in gemiste_spans.most_common(20):
    print(f"  {count:4d}x  '{span}'")

Totaal voorkomens van geannoteerde spans: 7541
Niet geannoteerde duplicaten:             1
Uitspraken met minstens één duplicaat:    1/1000
Gemiddeld extra per uitspraak:            0.0

Top 20 meest gemiste spans:
     1x  'paragraaf 8.2.2. van de Regeling'


In [61]:
for ecli in eclis[:4]:
    display(toon_annotaties(ecli, results, df_sample))